# 04 Backtests, Benchmarks, and Referee-Style Report

This notebook compares:

- Buy-and-hold per stock over weekend horizons
- Naive always-long weekend strategy
- Pure XGBoost threshold strategy
- PPO + XGBoost policy

It produces the final `strategy_summary` table and diagnostics.


## Introduction and Motivation

This exercise follows a skeptical, out-of-sample evaluation mindset common in empirical asset pricing and systematic trading research. We anchor design choices in:

- French (1980) on weekend/calendar return behavior.
- Friedman (2001) and Chen & Guestrin (2016) for boosting and XGBoost.
- Gu, Kelly, Xiu (2020) for ML return prediction context.
- Moody & Saffell (2001), Jiang et al. (2017) for RL-style trading frameworks.

The objective is **decision support**, not a claim of persistent alpha.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUT_DIR = PROJECT_ROOT / "research_outputs" / "weekend_rl_xgb"
TABLE_DIR = OUT_DIR / "tables"
FIG_DIR = OUT_DIR / "figures"
for p in [TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

xgb_panel = pd.read_parquet(OUT_DIR / "xgb_predictions_panel.parquet")
rl_log = pd.read_parquet(OUT_DIR / "rl_episode_log.parquet")

xgb_panel = xgb_panel.sort_values(["date_decision", "ticker"]).reset_index(drop=True)
rl_log = rl_log.sort_values("date_decision").reset_index(drop=True)


In [ ]:
def annualized_return(returns: pd.Series, periods_per_year: int = 52):
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan
    cum = (1 + returns).prod()
    years = len(returns) / periods_per_year
    return cum ** (1 / years) - 1 if years > 0 else np.nan


def annualized_vol(returns: pd.Series, periods_per_year: int = 52):
    return returns.std(ddof=1) * np.sqrt(periods_per_year)


def sharpe_ratio(returns: pd.Series, periods_per_year: int = 52):
    vol = annualized_vol(returns, periods_per_year)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return annualized_return(returns, periods_per_year) / vol


def max_drawdown(returns: pd.Series):
    nav = (1 + returns.fillna(0)).cumprod()
    peak = nav.cummax()
    dd = (peak - nav) / peak
    return dd.max()


def summarize_strategy(name: str, returns: pd.Series, turnover: pd.Series | None = None):
    out = {
        "strategy": name,
        "n_obs": int(returns.notna().sum()),
        "cum_return": float((1 + returns.fillna(0)).prod() - 1),
        "ann_return": float(annualized_return(returns)),
        "ann_vol": float(annualized_vol(returns)),
        "sharpe": float(sharpe_ratio(returns)),
        "max_drawdown": float(max_drawdown(returns)),
        "hit_rate": float((returns > 0).mean()),
    }
    if turnover is not None:
        out["avg_turnover"] = float(turnover.mean())
    return out


In [ ]:
# Date-level mapping for benchmark construction.
df_amzn = xgb_panel.loc[xgb_panel["ticker"] == "AMZN", ["date_decision", "y_true_weekend", "y_pred_xgb"]].rename(columns={"y_true_weekend": "ret_amzn", "y_pred_xgb": "pred_amzn"})
df_msft = xgb_panel.loc[xgb_panel["ticker"] == "MSFT", ["date_decision", "y_true_weekend", "y_pred_xgb"]].rename(columns={"y_true_weekend": "ret_msft", "y_pred_xgb": "pred_msft"})
df = df_amzn.merge(df_msft, on="date_decision", how="inner").sort_values("date_decision").reset_index(drop=True)

# Benchmarks.
df["ret_bh_amzn"] = df["ret_amzn"]
df["ret_bh_msft"] = df["ret_msft"]
df["ret_always_long_equal"] = 0.5 * (df["ret_amzn"] + df["ret_msft"])

# Pure XGBoost threshold rule.
thr = 0.0
df["w_xgb_amzn"] = (df["pred_amzn"] > thr).astype(float)
df["w_xgb_msft"] = (df["pred_msft"] > thr).astype(float)

# Normalize to max gross 1.0.
w_sum = df["w_xgb_amzn"] + df["w_xgb_msft"]
df["w_xgb_amzn_norm"] = np.where(w_sum > 0, df["w_xgb_amzn"] / w_sum, 0.0)
df["w_xgb_msft_norm"] = np.where(w_sum > 0, df["w_xgb_msft"] / w_sum, 0.0)

txn_cost = 5.0 / 10_000.0
df["turnover_xgb"] = (df[["w_xgb_amzn_norm", "w_xgb_msft_norm"]].diff().abs().sum(axis=1)).fillna(
    df["w_xgb_amzn_norm"].abs() + df["w_xgb_msft_norm"].abs()
)
df["ret_xgb_rule"] = (
    df["w_xgb_amzn_norm"] * df["ret_amzn"] +
    df["w_xgb_msft_norm"] * df["ret_msft"] -
    txn_cost * df["turnover_xgb"]
)

# RL strategy (already net of cost in notebook 03).
rl_series = rl_log[["date_decision", "net_ret", "turnover"]].rename(columns={"net_ret": "ret_rl", "turnover": "turnover_rl"})
df = df.merge(rl_series, on="date_decision", how="left")


In [ ]:
summary_rows = []
summary_rows.append(summarize_strategy("buy_hold_amzn", df["ret_bh_amzn"]))
summary_rows.append(summarize_strategy("buy_hold_msft", df["ret_bh_msft"]))
summary_rows.append(summarize_strategy("always_long_equal", df["ret_always_long_equal"]))
summary_rows.append(summarize_strategy("xgb_threshold_rule", df["ret_xgb_rule"], df["turnover_xgb"]))
summary_rows.append(summarize_strategy("ppo_xgb", df["ret_rl"], df["turnover_rl"]))

strategy_summary = pd.DataFrame(summary_rows)
strategy_summary


In [ ]:
# Regime-sliced analysis using lagged equal-weight realized sign as proxy.
df["market_proxy_prev"] = (0.5 * (df["ret_amzn"].shift(1) + df["ret_msft"].shift(1)) > 0).astype(int)

slice_rows = []
for regime, grp in df.groupby("market_proxy_prev"):
    lbl = "up_regime" if regime == 1 else "down_regime"
    slice_rows.append({
        "regime": lbl,
        "xgb_mean": grp["ret_xgb_rule"].mean(),
        "rl_mean": grp["ret_rl"].mean(),
        "xgb_hit": (grp["ret_xgb_rule"] > 0).mean(),
        "rl_hit": (grp["ret_rl"] > 0).mean(),
        "n": len(grp),
    })
regime_table = pd.DataFrame(slice_rows)
regime_table


In [ ]:
# Block bootstrap confidence intervals (weekly blocks).
def block_bootstrap_mean_ci(x: pd.Series, block=8, n_boot=1000, alpha=0.05, seed=42):
    rs = np.random.RandomState(seed)
    arr = x.dropna().values
    n = len(arr)
    if n == 0:
        return np.nan, np.nan
    idx_blocks = [np.arange(i, min(i + block, n)) for i in range(0, n, block)]
    means = []
    for _ in range(n_boot):
        picks = rs.randint(0, len(idx_blocks), size=len(idx_blocks))
        sample_idx = np.concatenate([idx_blocks[p] for p in picks])[:n]
        means.append(arr[sample_idx].mean())
    lo = np.quantile(means, alpha / 2)
    hi = np.quantile(means, 1 - alpha / 2)
    return float(lo), float(hi)

ci_rows = []
for name, series in {
    "xgb_threshold_rule": df["ret_xgb_rule"],
    "ppo_xgb": df["ret_rl"],
    "always_long_equal": df["ret_always_long_equal"],
}.items():
    lo, hi = block_bootstrap_mean_ci(series, block=8, n_boot=500)
    ci_rows.append({"strategy": name, "mean_ret": float(series.mean()), "ci_lo": lo, "ci_hi": hi})

bootstrap_ci = pd.DataFrame(ci_rows)
bootstrap_ci


In [ ]:
# Plots: cumulative returns.
plot_df = pd.DataFrame({
    "date": df["date_decision"],
    "always_long_equal": (1 + df["ret_always_long_equal"].fillna(0)).cumprod(),
    "xgb_threshold_rule": (1 + df["ret_xgb_rule"].fillna(0)).cumprod(),
    "ppo_xgb": (1 + df["ret_rl"].fillna(0)).cumprod(),
})

plt.figure(figsize=(11, 6))
for c in ["always_long_equal", "xgb_threshold_rule", "ppo_xgb"]:
    plt.plot(plot_df["date"], plot_df[c], label=c)
plt.legend()
plt.title("Weekend Strategy Equity Curves (Test Horizon)")
plt.xlabel("Decision Date")
plt.ylabel("Cumulative NAV")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR / "strategy_equity_curves.png", dpi=150)
plt.close()


In [ ]:
# Persist all report artifacts.
strategy_summary.to_csv(TABLE_DIR / "strategy_summary.csv", index=False)
regime_table.to_csv(TABLE_DIR / "regime_slice_summary.csv", index=False)
bootstrap_ci.to_csv(TABLE_DIR / "bootstrap_ci.csv", index=False)
df.to_parquet(OUT_DIR / "backtest_panel.parquet", index=False)

report_meta = {
    "sample_min": str(df["date_decision"].min().date()),
    "sample_max": str(df["date_decision"].max().date()),
    "n_obs": int(len(df)),
    "txn_cost_bps": 5.0,
    "threshold_xgb": 0.0,
    "notes": [
        "Two-asset universe (AMZN, MSFT) limits cross-sectional generalizability.",
        "Results are sensitive to transaction-cost assumptions and regime dependence.",
        "RL model risk remains high in small-sample, non-stationary settings.",
    ],
}
with open(OUT_DIR / "report_metadata.json", "w", encoding="utf-8") as f:
    json.dump(report_meta, f, indent=2)

strategy_summary


## Limitations and Extensions

- Small cross-section (2 equities) implies weak statistical power and high strategy variance.
- Weekend effects can be regime-dependent and may decay under crowding.
- PPO policy stability can vary with random seeds and reward shaping.
- Extensions: add broader universe, integrate realistic slippage/execution windows, and run nested walk-forward retraining for RL.
